In [1]:
import geopandas as gpd
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)
zones_mmt = gpd.read_file(f'{input_file_path}/network_agreg/mmt/ZonesMMT.shp')
zones_mmt = zones_mmt.to_crs(operation_crs)

# Spatial join
segments_mmt = gpd.sjoin(index_walkability, zones_mmt, how="inner", predicate="within")


In [2]:
mmt_walk = segments_mmt.groupby("MMT_NO")["walkability_index"].mean().reset_index()

# Merge back with zones_mmt polygons
zones_mmt = zones_mmt.merge(mmt_walk, on="MMT_NO", how="left")


In [3]:
zones_mmt

,MMT_NO,MMT_NOM,AIRE_URB_P,AIRE_URBAI,EPCI,CANT_DEP,COMMUNE,MAJ_ENTITE,X_COORD,Y_COORD,...,NOM_MACRO,PACA_PA4,PACA,NO_MACRO,NO_MEGA,Shape_Leng,Shape_Area,delta50_20,geometry,walkability_index
0,701010,Treulaz,Aire villageoise,Aire villageoise,Genève,Genève,Aire-la-Ville,2022-12-22,2.492065e+06,1.115864e+06,...,RA Rouge,Genève - Bernex - St-Julien,Rhône,13,3,4268.568091,7.432499e+05,7,"POLYGON ((2491248.582 1116033.748, 2491273.282...",0.285154
1,701029,Cheneviers,Aire villageoise,Aire villageoise,Genève,Genève,Aire-la-Ville,2022-12-22,2.491537e+06,1.117159e+06,...,RA Rouge,Genève - Bernex - St-Julien,Rhône,13,3,3754.001425,7.769210e+05,320,"POLYGON ((2491121.682 1116989.448, 2491129.782...",0.272465
2,701040,Fin & Vieux-Four,Aire villageoise,Aire villageoise,Genève,Genève,Aire-la-Ville,2022-12-22,2.492643e+06,1.116298e+06,...,RA Rouge,Genève - Bernex - St-Julien,Rhône,13,3,6072.024930,1.406278e+06,170,"POLYGON ((2491906.582 1116379.948, 2491966.082...",0.291448
3,702011,Anières-lac,Aire villageoise,Aire villageoise,Genève,Genève,Anières,2022-12-22,2.506223e+06,1.125919e+06,...,RG Lac Nord,Chablais,Chablais,6,5,4608.347722,4.435673e+05,164,"POLYGON ((2506527.582 1126367.048, 2506579.382...",0.258608
4,702012,Huttins,Aire villageoise,Aire villageoise,Genève,Genève,Anières,2022-12-22,2.506795e+06,1.126664e+06,...,RG Lac Nord,Chablais,Chablais,6,5,4144.409457,5.355112e+05,154,"POLYGON ((2506491.782 1126389.948, 2506499.182...",0.234391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
905,974782,Cluses-Les Ewues,Aire urbaine d'agglomération régionale,Aire urbaine d'agglomération régionale,CC Cluses-Arve et Montagnes,Haute-Savoie,74081,2022-12-22,2.532994e+06,1.102296e+06,...,CC Cluses-Arve et Montagnes,None,None,33,3002,2909.773107,2.238490e+05,1417,"POLYGON ((2532310.554 1102775.413, 2532355.706...",NaN
906,974783,Cluses-Les Villages,Aire urbaine d'agglomération régionale,Aire urbaine d'agglomération régionale,CC Cluses-Arve et Montagnes,Haute-Savoie,74081,2022-12-22,2.532899e+06,1.103244e+06,...,CC Cluses-Arve et Montagnes,None,None,33,3002,7862.060975,2.719310e+06,1518,"POLYGON ((2531567.837 1103530.247, 2531584.77 ...",NaN
907,974784,Cluses-Centre-Ville,Aire urbaine d'agglomération régionale,Aire urbaine d'agglomération régionale,CC Cluses-Arve et Montagnes,Haute-Savoie,74081,2022-12-22,2.533282e+06,1.101780e+06,...,CC Cluses-Arve et Montagnes,None,None,33,3002,6974.278849,1.554586e+06,2241,"POLYGON ((2532567.925 1101871.67, 2532573.238 ...",NaN
908,974785,Cluses-Messy,Aire urbaine d'agglomération régionale,Aire urbaine d'agglomération régionale,CC Cluses-Arve et Montagnes,Haute-Savoie,74081,2022-12-22,2.532430e+06,1.101841e+06,...,CC Cluses-Arve et Montagnes,None,None,33,3002,4717.383473,7.074778e+05,1007,"POLYGON ((2531669.93 1102468.72, 2531704.74 11...",NaN


In [4]:
#Length-weighted mean: longer segments should contribute more.
#segments_mmt["length"] = segments_mmt.geometry.length
#mmt_walk = (segments_mmt["walkability_index"] * segments_mmt["length"]).groupby(segments_mmt["MMT_NO"]).sum() / segments_mmt.groupby("MMT_NO")["length"].sum()


In [5]:
#Drop nan values 
zones_mmt = zones_mmt.dropna(subset=["walkability_index"])

In [6]:
#save the file 
zones_mmt.to_file(os.path.join(output_step3_path, "step3_aggregated_index_mmt.gpkg"), driver="GPKG")
zones_mmt.to_parquet(f'{output_step3_path}/step3_aggregated_index_mmt.parquet')